<a href="https://colab.research.google.com/github/Abusooma/Abusooma/blob/main/heart_attack_disease_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [74]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from kmodes.kmodes import KModes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
import warnings

**Charger les datasets**

In [75]:
df = pd.read_csv("/content/cardio_train.csv", sep=";")

In [76]:
df.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           70000 non-null  int64  
 1   age          70000 non-null  int64  
 2   gender       70000 non-null  int64  
 3   height       70000 non-null  int64  
 4   weight       70000 non-null  float64
 5   ap_hi        70000 non-null  int64  
 6   ap_lo        70000 non-null  int64  
 7   cholesterol  70000 non-null  int64  
 8   gluc         70000 non-null  int64  
 9   smoke        70000 non-null  int64  
 10  alco         70000 non-null  int64  
 11  active       70000 non-null  int64  
 12  cardio       70000 non-null  int64  
dtypes: float64(1), int64(12)
memory usage: 6.9 MB


Supprimer la colonne id unitile pour l'analyse

In [78]:
df.drop('id', axis=1, inplace=True)

In [79]:
df

,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,17474,1,156,56.0,100,60,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
69995,19240,2,168,76.0,120,80,1,1,1,0,1,0
69996,22601,1,158,126.0,140,90,2,2,0,0,1,1
69997,19066,2,183,105.0,180,90,3,1,0,1,0,1
69998,22431,1,163,72.0,135,80,1,2,0,0,0,1


In [80]:
df.shape

(70000, 12)

Suppression des Outliers dans le dataset

In [81]:
print(f"Les lignes avant suppression des outliers: {df.shape[0]}")

cols_for_outlier_removal = ['ap_hi', 'ap_lo', 'weight', 'height']
for col in cols_for_outlier_removal:
    lower_quantile = df[col].quantile(0.025)
    upper_quantile = df[col].quantile(0.975)
    df = df[(df[col] >= lower_quantile) & (df[col] <= upper_quantile)]


print(f"Les lignes avant suppression des outliers {df.shape[0]}")


Les lignes avant suppression des outliers: 70000
Les lignes avant suppression des outliers 60752


# --- Feature Engineering and Transformation ---

Convert 'age' from days to years

In [82]:
df['age_in_years'] = (df['age'] / 365).round().astype(int)

In [83]:
df

,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_in_years
0,18393,2,168,62.0,110,80,1,1,0,0,1,0,50
1,20228,1,156,85.0,140,90,3,1,0,0,1,1,55
2,18857,1,165,64.0,130,70,3,1,0,0,0,1,52
3,17623,2,169,82.0,150,100,1,1,0,0,1,1,48
4,17474,1,156,56.0,100,60,1,1,0,0,0,0,48
...,...,...,...,...,...,...,...,...,...,...,...,...,...
69993,19699,1,172,70.0,130,90,1,1,0,0,1,1,54
69994,21074,1,165,80.0,150,80,1,1,0,0,1,1,58
69995,19240,2,168,76.0,120,80,1,1,1,0,1,0,53
69998,22431,1,163,72.0,135,80,1,2,0,0,0,1,61


##### Nous créons la variable "bmi" Indice de Masse Corporelle (IMC).
##### L’IMC est calculé comme le poids divisé par la taille au carré (en mètres).
##### C’est un indicateur largement utilisé pour évaluer si une personne a un poids santé par rapport à sa taille et il est important dans l’analyse du risque cardiovasculaire

In [84]:
df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)

In [85]:
df

,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_in_years,bmi
0,18393,2,168,62.0,110,80,1,1,0,0,1,0,50,21.967120
1,20228,1,156,85.0,140,90,3,1,0,0,1,1,55,34.927679
2,18857,1,165,64.0,130,70,3,1,0,0,0,1,52,23.507805
3,17623,2,169,82.0,150,100,1,1,0,0,1,1,48,28.710479
4,17474,1,156,56.0,100,60,1,1,0,0,0,0,48,23.011177
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69993,19699,1,172,70.0,130,90,1,1,0,0,1,1,54,23.661439
69994,21074,1,165,80.0,150,80,1,1,0,0,1,1,58,29.384757
69995,19240,2,168,76.0,120,80,1,1,1,0,1,0,53,26.927438
69998,22431,1,163,72.0,135,80,1,2,0,0,0,1,61,27.099251


**We create a new feature called Mean Arterial Pressure (MAP).  
MAP provides a better overall measure of blood pressure than just systolic (ap_hi)
or diastolic (ap_lo) values alone, because it accounts for the fact that the heart
spends more time in diastole.  
This feature is often used in medical analysis as it reflects the average pressure
experienced by the organs and can be more informative for cardiovascular risk prediction.**

In [86]:
df['map'] = ((2 * df['ap_lo']) + df['ap_hi']) / 3

In [87]:
df

,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_in_years,bmi,map
0,18393,2,168,62.0,110,80,1,1,0,0,1,0,50,21.967120,90.000000
1,20228,1,156,85.0,140,90,3,1,0,0,1,1,55,34.927679,106.666667
2,18857,1,165,64.0,130,70,3,1,0,0,0,1,52,23.507805,90.000000
3,17623,2,169,82.0,150,100,1,1,0,0,1,1,48,28.710479,116.666667
4,17474,1,156,56.0,100,60,1,1,0,0,0,0,48,23.011177,73.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69993,19699,1,172,70.0,130,90,1,1,0,0,1,1,54,23.661439,103.333333
69994,21074,1,165,80.0,150,80,1,1,0,0,1,1,58,29.384757,103.333333
69995,19240,2,168,76.0,120,80,1,1,1,0,1,0,53,26.927438,93.333333
69998,22431,1,163,72.0,135,80,1,2,0,0,0,1,61,27.099251,98.333333


**We discretize age into 5-year intervals (30–35, 35–40, …, 60–65).  
Grouping age into bins makes it easier to analyze trends across age ranges
and is consistent with the approach described in the paper.  
This feature helps highlight age-related risk categories instead of treating
age as a continuous variable.**

In [88]:
age_bins = [30, 35, 40, 45, 50, 55, 60, 65]
age_labels = list(range(7))
df['age_bin'] = pd.cut(df['age_in_years'], bins=age_bins, labels=age_labels, right=False, include_lowest=True)

We convert BMI into categorical classes using the standard WHO BMI categories.  
This allows us to better capture weight-related health risks (underweight, normal, overweight, obesity levels)
and makes the feature more interpretable for medical analysis.


In [89]:
bmi_bins = [0, 18.5, 25, 30, 35, 40, np.inf]
bmi_labels = [0, 1, 2, 3, 4, 5]
df['BMI_Class'] = pd.cut(df['bmi'], bins=bmi_bins, labels=bmi_labels, right=False)

We discretize Mean Arterial Pressure (MAP) into categories with fixed ranges (70–80, 80–90, …).  
Categorizing MAP helps identify clinically meaningful blood pressure ranges
and aligns with the intervals specified in the reference paper.


In [90]:
map_bins = [0, 70, 80, 90, 100, 110, np.inf]
map_labels = [0, 1, 2, 3, 4, 5]
df['MAP_Class'] = pd.cut(df['map'], bins=map_bins, labels=map_labels, right=False)

In [91]:
# Drop original and intermediate columns
df_final = df.drop(['age', 'height', 'weight', 'ap_hi', 'ap_lo', 'age_in_years', 'bmi', 'map'], axis=1)

In [92]:
# Rearrange columns to match the final attributes in Table 4 of the paper
# The target variable 'cardio' is moved to the end.
final_cols = ['gender', 'age_bin', 'BMI_Class', 'MAP_Class', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio']
df_final = df_final[final_cols]
df_final.dropna(inplace=True) # Drop rows with NaN values that might arise from binning

In [93]:
df_final.head()

,gender,age_bin,BMI_Class,MAP_Class,cholesterol,gluc,smoke,alco,active,cardio
0,2,4,1,3,1,1,0,0,1,0
1,1,5,3,4,3,1,0,0,1,1
2,1,4,1,3,3,1,0,0,0,1
3,2,3,2,5,1,1,0,0,1,1
4,1,3,1,1,1,1,0,0,0,0


In [94]:
df.shape

(60752, 18)

In [95]:
#  Clustering (as per paper Section 3.4) ---
print("Applying K-Modes Clustering ---")

# 4.1. Split dataset by gender
df_male = df_final[df_final['gender'] == 2].copy()
df_female = df_final[df_final['gender'] == 1].copy()

Applying K-Modes Clustering ---


In [96]:
k = 2

km_male = KModes(n_clusters=k, init='Huang', n_init=5, random_state=42)
df_male['cluster'] = km_male.fit_predict(df_male)

km_female = KModes(n_clusters=k, init='Huang', n_init=5, random_state=42)
df_female['cluster'] = km_female.fit_predict(df_female)

df_clustered = pd.concat([df_male, df_female], ignore_index=True)

#### Data shape after clustering and combinin

In [97]:
df_clustered.shape

(60439, 11)

#### Data Head after Adding Cluster Feature

In [98]:
df_clustered.head()

,gender,age_bin,BMI_Class,MAP_Class,cholesterol,gluc,smoke,alco,active,cardio,cluster
0,2,4,1,3,1,1,0,0,1,0,0
1,2,3,2,5,1,1,0,0,1,1,1
2,2,6,2,4,3,3,0,0,1,1,1
3,2,4,1,3,1,1,0,0,1,0,0
4,2,2,1,3,1,1,0,0,0,0,0


# Model Training and Evaluation

##### Prepare data for modeling

In [99]:
X = df_clustered.drop('cardio', axis=1)
y = df_clustered['cardio']
for col in ['age_bin', 'BMI_Class', 'MAP_Class']:
    X[col] = X[col].astype(int)

##### Split data into training and testing sets (80:20 ratio)

In [100]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set size: {X_train.shape[0]}, Testing set size: {X_test.shape[0]}")

Training set size: 48351, Testing set size: 12088


In [103]:
# Define models and hyperparameter grids for GridSearchCV
models = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Multilayer Perceptron': MLPClassifier(random_state=42, max_iter=1000),
    'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
}

params = {
    'Random Forest': {
        'n_estimators': [100, 200],
        'max_depth': [10, 20],
        'min_samples_leaf': [1, 5]
    },
    'Multilayer Perceptron': {
        'hidden_layer_sizes': [(50, 50), (100,)],
        'activation': ['tanh', 'relu'],
        'solver': ['adam'],
        'alpha': [0.0001, 0.05],
    },
    'XGBoost': {
        'n_estimators': [100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1]
    }
}

#  Run GridSearchCV and evaluate each model
results = []

for model_name in models:
    print(f"\n--- Tuning and Evaluating {model_name} ---")

    # Initialize GridSearchCV
    # The paper uses k-fold cross-validation. We use 5 folds as a standard.
    grid_search = GridSearchCV(models[model_name], params[model_name], cv=5, scoring='accuracy', n_jobs=-1, verbose=1)

    # Fit the model
    grid_search.fit(X_train, y_train)

    # Get the best estimator
    best_model = grid_search.best_estimator_

    print(f"Best Parameters for {model_name}: {grid_search.best_params_}")

    # Make predictions
    y_pred = best_model.predict(X_test)
    y_pred_proba = best_model.predict_proba(X_test)[:, 1]

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)

    # Store results
    results.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'AUC': auc,
        'Best CV Score': grid_search.best_score_
    })

    print(f"\n--- {model_name} Performance on Test Set ---")
    print(classification_report(y_test, y_pred))

    # # Plot ROC Curve
    # fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    # plt.figure()
    # plt.plot(fpr, tpr, label=f'{model_name} (AUC = {auc:.2f})')
    # plt.plot([0, 1], [0, 1], 'k--')
    # plt.xlabel('False Positive Rate')
    # plt.ylabel('True Positive Rate')
    # plt.title(f'ROC Curve for {model_name}')
    # plt.legend(loc='best')
    # plt.show()


--- Tuning and Evaluating Random Forest ---
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best Parameters for Random Forest: {'max_depth': 20, 'min_samples_leaf': 5, 'n_estimators': 100}

--- Random Forest Performance on Test Set ---
              precision    recall  f1-score   support

           0       0.82      0.91      0.86      6154
           1       0.90      0.79      0.84      5934

    accuracy                           0.85     12088
   macro avg       0.86      0.85      0.85     12088
weighted avg       0.86      0.85      0.85     12088


--- Tuning and Evaluating Multilayer Perceptron ---
Fitting 5 folds for each of 8 candidates, totalling 40 fits


KeyboardInterrupt: 

# --- Final Results Summary ---
#### Summary of Model Performance

In [102]:
results_df = pd.DataFrame(results)

In [ ]:
results_df